In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import seaborn as sns
import textwrap

# ==============================================================================
# BLOK 0: KONFIGURACJA BADANIA (SETUP)
# ==============================================================================
# Parametry modelu
HIPOTEZA = "H1: Rozwój OZE w Niemczech wykazuje trend deterministyczny, który może zostać trwale zakłócony jedynie przez silne szoki strukturalne w otoczeniu makroekonomicznym."
ZMIENNA_OBJASNIANA = 'res_share'  # Y
ZMIENNE_OBJASIAJACE = ['time_index'] # X (w modelu trendu)

# Wczytanie i czyszczenie danych
try:
    df = pd.read_csv('dane_miesieczne_DE.csv')
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
    df = df.sort_values('date')
    df['time_index'] = np.arange(len(df))
except FileNotFoundError:
    # Generowanie sztucznych danych (zabezpieczenie)
    print("CRITICAL ERROR: Brak pliku danych. Generuję dane testowe...")
    dates = pd.date_range(start='2015-01-01', periods=120, freq='M')
    df = pd.DataFrame({
        'date': dates,
        'res_share': np.linspace(0.3, 0.6, 120) + np.random.normal(0, 0.02, 120),
        'fossil_share': np.linspace(0.6, 0.3, 120) + np.random.normal(0, 0.02, 120),
        'market_price': np.random.uniform(20, 100, 120),
        'energy_consumption': np.random.uniform(40, 60, 120),
        'time_index': np.arange(120)
    })

# ==============================================================================
# BLOK 1: MODELOWANIE EKONOMETRYCZNE (PIOTR - SZEF TECHNICZNY)
# ==============================================================================
# 1.1. Estymacja klasyczną Metodą Najmniejszych Kwadratów (MNK/OLS)
X = df[['time_index']]
y = df[ZMIENNA_OBJASNIANA]

model = LinearRegression()
model.fit(X, y)

# 1.2. Diagnostyka modelu
y_pred = model.predict(X)
residuals = y - y_pred
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

# Parametry trendu
slope = model.coef_[0]
intercept = model.intercept_

In [2]:
# ==============================================================================
# BLOK 2: GENEROWANIE SCENARIUSZY (SYMULACJA SZOKÓW)
# ==============================================================================
# Prognoza ex-ante (do 2030)
last_date = df['date'].iloc[-1]
last_index = df['time_index'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), end='2030-12-31', freq='M')
future_X = np.arange(last_index + 1, last_index + 1 + len(future_dates)).reshape(-1, 1)

# Scenariusz Bazowy
forecast_base = model.predict(future_X)

# Definicja czynników zakłócających
factors_data = {
    'Sfera': ['Technologia', 'Makroekonomia', 'Geopolityka', 'Klimat', 'Legislacja'],
    'Szok': [
        'Przełom w H2 (Wodór)',
        'Stagflacja w UE',
        'Kryzys surowcowy (węgiel)',
        'Anomalie pogodowe',
        'Deregulacja OZE'
    ],
    'Kierunek': [1, -1, -1, 1, 1],
    'Waga_Ekspercka': [5, 4, 3, 4, 2],
    'Prawdopodobienstwo': [0.20, 0.40, 0.30, 0.60, 0.50]
}
factors_df = pd.DataFrame(factors_data)

# Obliczenie "Impact Factor"
factors_df['Impact_Factor'] = factors_df['Waga_Ekspercka'] * factors_df['Prawdopodobienstwo'] * factors_df['Kierunek']

total_positive_shock = factors_df[factors_df['Impact_Factor'] > 0]['Impact_Factor'].sum()
total_negative_shock = factors_df[factors_df['Impact_Factor'] < 0]['Impact_Factor'].sum()

# Kalibracja wrażliwości modelu
sensitivity_param = slope * 0.15

forecast_opt = forecast_base + (future_X.flatten() - last_index) * (total_positive_shock * sensitivity_param)
forecast_pes = forecast_base + (future_X.flatten() - last_index) * (total_negative_shock * sensitivity_param)

/tmp/ipython-input-884671108.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), end='2030-12-31', freq='M')
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [3]:

# ==============================================================================
# BLOK 3: OUTPUT DLA ZESPOŁU (KARTY PRACY)
# ==============================================================================

print("="*60)
print("RAPORT DIAGNOSTYCZNY - DO ROZDZIELENIA ZADAŃ")
print("="*60)

print(f"\n[DLA PIOTRA] - Sekcja Metodologiczna i Diagnostyka")
print(f"1. Model: Y = {intercept:.4f} + {slope:.4f} * t + e")
print(f"2. Dopasowanie (R^2): {r2:.4f} (Model wyjaśnia {r2*100:.1f}% zmienności historycznej)")
print(f"3. Błąd standardowy (RMSE): {rmse:.4f}")
print("   Zadanie: Opisz, czy reszty mają rozkład normalny (patrz wykres w PDF).")
print("   Jeśli reszty układają się w wzór, model może pomijać zmienną cykliczną.")

print(f"\n[DLA IZY] - Sekcja Makroekonomiczna i Kontekst")
print("   Zadanie: Przeanalizuj tabelę korelacji w PDF.")
print("   Pytanie badawcze: Dlaczego korelacja OZE z ceną jest niska/wysoka?")
print("   Hipoteza do weryfikacji: Czy 'Merit Order Effect' (tanie OZE obniża ceny) jest widoczny w danych?")
print(f"   Dane pomocnicze: Średnia cena historyczna = {df['market_price'].mean():.2f} EUR/MWh")

print(f"\n[DLA KLAUDII] - Sekcja Strategiczna (Foresight)")
print("   Zadanie: Uzasadnij dobór wag w tabeli czynników zakłócających.")
print("   Pytanie badawcze: Który z tych czynników jest 'Czarnym Łabędziem' (małe prawd., duży wpływ)?")
print(f"   Wynik modelu: Scenariusz optymistyczny zakłada przyspieszenie o {(total_positive_shock * sensitivity_param)/slope:.1%} względem trendu.")
print("="*60)



RAPORT DIAGNOSTYCZNY - DO ROZDZIELENIA ZADAŃ

[DLA PIOTRA] - Sekcja Metodologiczna i Diagnostyka
1. Model: Y = 0.1915 + 0.0010 * t + e
2. Dopasowanie (R^2): 0.6231 (Model wyjaśnia 62.3% zmienności historycznej)
3. Błąd standardowy (RMSE): 0.0302
   Zadanie: Opisz, czy reszty mają rozkład normalny (patrz wykres w PDF).
   Jeśli reszty układają się w wzór, model może pomijać zmienną cykliczną.

[DLA IZY] - Sekcja Makroekonomiczna i Kontekst
   Zadanie: Przeanalizuj tabelę korelacji w PDF.
   Pytanie badawcze: Dlaczego korelacja OZE z ceną jest niska/wysoka?
   Hipoteza do weryfikacji: Czy 'Merit Order Effect' (tanie OZE obniża ceny) jest widoczny w danych?
   Dane pomocnicze: Średnia cena historyczna = 72.54 EUR/MWh

[DLA KLAUDII] - Sekcja Strategiczna (Foresight)
   Zadanie: Uzasadnij dobór wag w tabeli czynników zakłócających.
   Pytanie badawcze: Który z tych czynników jest 'Czarnym Łabędziem' (małe prawd., duży wpływ)?
   Wynik modelu: Scenariusz optymistyczny zakłada przyspieszenie 

In [4]:
# ==============================================================================
# BLOK 4: GENEROWANIE RAPORTU PDF (WERSJA Z POPRAWIONYM LAYOUTEM)
# ==============================================================================
pdf_filename = 'Badanie_Scenariuszowe_OZE_2030_Raport_v2.pdf'
A4_SIZE_VERT = (8.27, 11.69)

with PdfPages(pdf_filename) as pdf:

    # ==========================================================================
    # STRONA 1: TYTUŁOWA
    # ==========================================================================
    fig = plt.figure(figsize=A4_SIZE_VERT)
    ax = fig.add_subplot(111)
    ax.axis('off')

    ax.text(0.5, 0.9, "ZADANIE 2", ha='center', fontsize=24, weight='bold', color='gray')
    ax.text(0.5, 0.6, "ANALIZA SCENARIUSZOWA\nROZWOJU OZE W NIEMCZECH\nPERSPEKTYWA 2030",
            ha='center', va='center', fontsize=28, weight='bold')
    ax.text(0.5, 0.45, textwrap.fill(HIPOTEZA, width=50),
            ha='center', va='center', fontsize=12, style='italic')
    ax.text(0.5, 0.2, "AUTORZY:\nPiotr Wiśniewski)\nIzabela Reszka\nKlaudia Woźniak",
            ha='center', fontsize=14)

    pdf.savefig()
    plt.close()

    # ==========================================================================
    # STRONA 2: DIAGNOSTYKA MODELU (Zmniejszone wykresy)
    # ==========================================================================
    fig = plt.figure(figsize=A4_SIZE_VERT)
    plt.suptitle("CZĘŚĆ 1: DIAGNOSTYKA MODELU", fontsize=16, weight='bold', y=0.95)

    # ZMIANA: Zmniejszono ratio dla wykresów (1.0, 1.0) względem tekstu i dodano hspace
    # height_ratios=[1, 1, 0.8] sprawi, że wykresy będą niższe
    gs = fig.add_gridspec(3, 1, height_ratios=[1, 1, 0.8], hspace=0.3)

    # Wykres 1
    ax1 = fig.add_subplot(gs[0])
    ax1.scatter(df['date'], df['res_share'], s=10, color='black', alpha=0.5, label='Obserwacje emp.')
    ax1.plot(df['date'], y_pred, color='blue', linewidth=2, label='Model teoretyczny (OLS)')
    ax1.set_title(f"Dopasowanie trendu liniowego (R2={r2:.2f})")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylabel("Udział OZE")

    # Wykres 2
    ax2 = fig.add_subplot(gs[1])
    sns.histplot(residuals, kde=True, ax=ax2, color='purple')
    ax2.set_title("Rozkład reszt modelu (Analiza stacjonarności błędów)")
    ax2.set_xlabel("Wartość reszty")
    ax2.set_ylabel("Liczebność")

    # Tekst
    ax3 = fig.add_subplot(gs[2])
    ax3.axis('off')
    tech_text = (
        f"METODOLOGIA I WNIOSKI TECHNICZNE:\n\n"
        f"Zastosowano model regresji liniowej względem czasu (OLS). "
        f"Estymator kierunkowy wynosi {slope:.5f}, co oznacza średni miesięczny przyrost "
        f"udziału OZE o ok. {slope*100:.2f} p.p. (ceteris paribus).\n\n"
        f"Współczynnik R2 na poziomie {r2:.2f} wskazuje na silną deterministyczną "
        f"składową procesu. Analiza histogramu reszt pozwala ocenić, czy w danych występują "
        f"zjawiska nieliniowe nieujęte w modelu."
    )
    ax3.text(0.05, 0.9, tech_text, fontsize=11, va='top', ha='left', fontfamily='sans-serif', wrap=True)

    pdf.savefig()
    plt.close()

    # ==========================================================================
    # STRONA 3: ANALIZA MAKRO - CZĘŚĆ 1 (Większy odstęp dla tekstu)
    # ==========================================================================
    fig = plt.figure(figsize=A4_SIZE_VERT)
    plt.suptitle("CZĘŚĆ 2: OTOCZENIE MAKROEKONOMICZNE (I)", fontsize=16, weight='bold', y=0.95)

    # ZMIANA: Zwiększono hspace (odstęp) między wykresem a tekstem
    gs = fig.add_gridspec(2, 1, height_ratios=[1.2, 1], hspace=0.4)

    # Macierz korelacji
    ax1 = fig.add_subplot(gs[0])
    corr_vars = ['res_share', 'fossil_share', 'market_price', 'energy_consumption']
    mask = np.triu(np.ones_like(df[corr_vars].corr(), dtype=bool))
    sns.heatmap(df[corr_vars].corr(), mask=mask, annot=True, cmap='RdBu',
                center=0, ax=ax1, square=True, cbar_kws={"shrink": .7})
    ax1.set_title("Macierz Korelacji Pearsona")

    # Tekst Część 1 - ZMIANA: tekst przesunięty w dół (y=0.8)
    ax2 = fig.add_subplot(gs[1])
    ax2.axis('off')

    macro_text_part1 = (
        "Na podstawie macierzy korelacji Pearsona dla danych miesięcznych z lat 2015–2025 "
        "sformułowano następujące wnioski dotyczące struktury niemieckiego rynku energii:\n\n"
        "1. Transformacja strukturalna i substytucyjność (korelacja: -0.96)\n"
        "Występuje niemal pełna, silna korelacja ujemna pomiędzy udziałem OZE (res_share) "
        "a udziałem paliw kopalnych (fossil_share). Wynik -0.96 potwierdza, że niemiecka "
        "transformacja energetyczna (Energiewende) nie polega jedynie na dodawaniu nowych mocy, "
        "lecz na bezpośredniej substytucji. Każdy wzrost produkcji z OZE niemal w relacji 1:1 "
        "wypiera generację konwencjonalną (węgiel, gaz)."
    )
    # y=0.8 zapewnia większy margines od góry subplota
    ax2.text(0.05, 0.8, macro_text_part1, fontsize=11, va='top', ha='left', fontfamily='serif', wrap=True)

    pdf.savefig()
    plt.close()

    # ==========================================================================
    # STRONA 4: ANALIZA MAKRO - CZĘŚĆ 2
    # ==========================================================================
    fig = plt.figure(figsize=A4_SIZE_VERT)
    plt.suptitle("CZĘŚĆ 2: OTOCZENIE MAKROEKONOMICZNE (II)", fontsize=16, weight='bold', y=0.95)

    ax_text = fig.add_subplot(111)
    ax_text.axis('off')

    macro_text_part2 = (
        "2. Weryfikacja efektu Merit Order (korelacja: +0.15)\n"
        "Teoria efektu Merit Order zakłada, że źródła OZE o zerowym koszcie krańcowym "
        "powinny obniżać hurtowe ceny energii. Tymczasem obserwowana jest słaba dodatnia "
        "korelacja pomiędzy udziałem OZE a ceną rynkową (market_price).\n"
        "W analizowanym okresie, obejmującym kryzys energetyczny 2021–2023, mechanizm "
        "obniżania cen przez OZE został zdominowany przez zewnętrzne szoki podażowe, "
        "w szczególności gwałtowny wzrost cen gazu ziemnego oraz uprawnień do emisji CO2.\n\n"

        "3. Efektywność energetyczna a OZE (korelacja: -0.62)\n"
        "Zauważalna jest istotna ujemna korelacja pomiędzy udziałem OZE "
        "a całkowitym zużyciem energii. Wynik ten odzwierciedla zarówno sezonowość "
        "produkcji energii odnawialnej, jak i długoterminowy trend redukcji popytu "
        "wymuszony wysokimi cenami energii oraz poprawą efektywności energetycznej.\n\n"

        "PODSUMOWANIE:\n"
        "Zmienne wykazują spójne powiązania fundamentalne. Jednocześnie dodatnia korelacja "
        "między udziałem OZE a ceną energii stanowi istotny sygnał ostrzegawczy, "
        "wskazujący na podatność rynku na szoki surowcowe, które mogą czasowo niwelować "
        "korzyści cenowe wynikające z transformacji energetycznej."
    )

    ax_text.text(0.05, 0.95, macro_text_part2, fontsize=11, va='top', ha='left', fontfamily='serif', wrap=True)

    pdf.savefig()
    plt.close()

    # ==========================================================================
    # STRONA 5: SCENARIUSZE I FORESIGHT (Zmniejszony wykres lejka)
    # ==========================================================================
    fig = plt.figure(figsize=A4_SIZE_VERT)
    plt.suptitle("CZĘŚĆ 3: PROGNOZA SCENARIUSZOWA (FORESIGHT)", fontsize=16, weight='bold', y=0.95)

    # ZMIANA: Ratio 1.8 do 1 (zamiast 3 do 1) - wykres będzie relatywnie niższy
    gs = fig.add_gridspec(2, 1, height_ratios=[1.8, 1])

    # Wykres Lejka
    ax = fig.add_subplot(gs[0])
    ax.plot(df['date'], df['res_share'], color='black', alpha=0.6, label='Dane Historyczne')
    dates_future_plot = future_dates
    ax.plot(dates_future_plot, forecast_base, color='blue', linestyle='--', linewidth=2, label='BAZOWY (Status Quo)')
    ax.plot(dates_future_plot, forecast_opt, color='green', linewidth=2, label='OPTYMISTYCZNY')
    ax.plot(dates_future_plot, forecast_pes, color='red', linewidth=2, label='PESYMISTYCZNY')
    ax.fill_between(dates_future_plot, forecast_pes, forecast_opt, color='gray', alpha=0.15, label='Stożek Niepewności')

    ax.set_title("Projekcja udziału OZE w miksie energetycznym Niemiec do 2030 r.")
    ax.set_ylabel("Udział OZE")
    ax.legend(loc='upper left')
    ax.grid(True, linestyle=':', alpha=0.7)

    # Tabela
    ax_table = fig.add_subplot(gs[1])
    ax_table.axis('off')

    table_text_header = "ZAŁOŻENIA SZOKÓW EGZOGENICZNYCH (Ekspert: Klaudia):"
    table_content = "\n".join([f"• {row['Szok']} (Waga: {row['Waga_Ekspercka']}, Prawdop.: {row['Prawdopodobienstwo']})" for i, row in factors_df.iterrows()])
    final_table_text = f"{table_text_header}\n\n{table_content}"

    # Tekst tabeli nieco wyżej (0.9), bo wykres jest mniejszy
    ax_table.text(0.05, 0.9, final_table_text, fontsize=10, verticalalignment='top',
                  bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=1.0, edgecolor='gray'))

    pdf.savefig()
    plt.close()

print(f"\nGenerowanie zakończone. Pobierz nowy plik: {pdf_filename}")


Generowanie zakończone. Pobierz nowy plik: Badanie_Scenariuszowe_OZE_2030_Raport_v2.pdf
